# Data analysis, session 2A: SQL and data quality
**October 15, 9:00-10:25 (85 minutes).** A **10-minute break** follows.

| Time | Activity |
|---|---|
| 9:00-9:10 | Recap; CSV and nested JSON |
| 9:10-9:45 | SELECT, WHERE, GROUP BY, JOIN; compare pandas results |
| 9:45-10:15 | OHLC checks, rolling outliers, documented repair |
| 10:15-10:25 | Inspect homework data; practice query and debrief |

Question: *How do we retrieve just the data we need and decide whether it can support a report?*
This module starts independently. Download `data.db` from bCourses into this lecture's `data/` folder.
We read the supplied database in read-only mode and write demonstrations to a separate output database.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sqlite3
from pathlib import Path
from IPython.display import display
%matplotlib inline
course_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file())
lesson_dir = course_root / 'Lectures/Lecture 4'
plt.style.use('seaborn-v0_8-whitegrid')

## 1. File ingestion is a schema decision
Inspect types, units, and dates after loading. CSV stores a table; an API's JSON may wrap the table in metadata.
The supplied snapshots are historical. Optional ingestion switches and CTE examples remain in [ingestion reference](reference/2_data_ingestion_and_sql.ipynb).

In [ ]:
import json
caps = pd.read_csv(lesson_dir / 'data/market_caps.csv', dtype={'ticker': 'string'})
display(caps.head())
payload = json.loads((lesson_dir / 'data/payload.json').read_text())
daily = pd.DataFrame.from_dict(payload['Time Series (Daily)'], orient='index')
daily.columns = ['open', 'high', 'low', 'close', 'volume']
daily.index = pd.to_datetime(daily.index)
daily = daily.apply(pd.to_numeric).sort_index()
display(daily.head())

## 2. Connect and inspect before querying
`mode=ro` prevents accidentally creating an empty database at a misspelled path.
The connection is closed at the end of this notebook. SQL operates on tables; pandas receives a query's result.

In [ ]:
db_path = lesson_dir / 'data/data.db'
if not db_path.is_file():
    raise FileNotFoundError(f'Download data.db from bCourses and place it at {db_path}')
conn = sqlite3.connect(db_path.resolve().as_uri() + '?mode=ro', uri=True)
display(pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn))
display(pd.read_sql_query('PRAGMA table_info(ohlc)', conn))

### Filter early and bind parameters
The `?` placeholders bind **values**, avoiding quoting mistakes and SQL injection. Table/column identifiers are fixed in our examples.
`ORDER BY` is required before calculating sequential returns; SQL does not promise a default row order.

In [ ]:
ticker = 'AAPL'
start_date, end_date = '2024-01-01', '2024-06-30'
query = """
SELECT ts, ticker, open, high, low, close, adj_close, volume
FROM ohlc
WHERE ticker = ? AND ts BETWEEN ? AND ?
ORDER BY ts
"""
aapl = pd.read_sql_query(query, conn, params=(ticker, start_date, end_date), parse_dates=['ts'])
assert not aapl.empty
print(f'{ticker}: {len(aapl)} observations')
display(aapl.head())

### GROUP BY and pandas should agree
`WHERE` filters observations before aggregation; `HAVING` filters groups after aggregation.
`COUNT(*)` counts rows, while `COUNT(close)` excludes null close values.

In [ ]:
sql_summary = pd.read_sql_query("""
SELECT ticker, COUNT(*) AS num_days, AVG(close) AS avg_close, AVG(volume) AS avg_volume
FROM ohlc WHERE ts BETWEEN ? AND ?
GROUP BY ticker HAVING COUNT(*) >= 100
ORDER BY avg_volume DESC
""", conn, params=(start_date, end_date))
display(sql_summary.head())
row = sql_summary.set_index('ticker').loc['AAPL']
assert row['num_days'] == len(aapl)
assert np.isclose(row['avg_close'], aapl['close'].mean())

### JOIN on the full key
Market capitalization is dated. Joining only on ticker would combine each daily price with every cap observation for that ticker.
Here a left join preserves trading days and leaves nulls on dates without a matching cap snapshot. Discuss why an inner join has fewer rows.

In [ ]:
joined = pd.read_sql_query("""
SELECT o.ts, o.ticker, o.close, m.market_capitalization / 1e9 AS market_cap_bn
FROM ohlc AS o
LEFT JOIN market_caps AS m ON o.ticker = m.ticker AND o.ts = m.ts
WHERE o.ticker = ? AND o.ts BETWEEN ? AND ?
ORDER BY o.ts
""", conn, params=(ticker, start_date, end_date), parse_dates=['ts'])
assert len(joined) == len(aapl), 'Unexpected duplicate keys on the right side.'
display(joined.loc[joined['market_cap_bn'].notna()].head())
print('Unmatched cap dates:', joined['market_cap_bn'].isna().sum())

## 3. Quality checks: definite faults versus review flags
Use raw OHLC columns for bounds checks, because adjusted close can be on a different scale.
A missing/non-finite price, negative volume, duplicate date, or impossible OHLC relationship fails our report contract.
The function below expects **one ticker** with dates as its index.

In [ ]:
def quality_flags(frame):
    """Flag missing/non-finite values, nonpositive equity prices, bad OHLC, and duplicate dates."""
    price_cols = ['open', 'high', 'low', 'close', 'adj_close']
    flags = pd.DataFrame(index=frame.index)
    flags['missing_or_nonfinite'] = ~np.isfinite(frame[price_cols + ['volume']]).all(axis=1)
    flags['nonpositive_price'] = frame[price_cols].le(0).any(axis=1)
    flags['ohlc_bounds'] = (frame['high'].lt(frame[['open', 'close']].max(axis=1))
                            | frame['low'].gt(frame[['open', 'close']].min(axis=1))
                            | frame['high'].lt(frame['low']))
    flags['negative_volume'] = frame['volume'].lt(0)
    flags['duplicate_date'] = frame.index.duplicated(keep=False)
    return flags

In [ ]:
original = aapl.set_index('ts').copy()
assert not quality_flags(original).any().any()
# Controlled exercise: only close is corrupted; other OHLC and adj_close are retained.
messy = original.copy()
messy.iloc[10, messy.columns.get_loc('close')] *= 10
messy.iloc[25, messy.columns.get_loc('close')] = np.nan
messy.iloc[40, messy.columns.get_loc('volume')] = -1
flags = quality_flags(messy)
display(flags.sum())
display(messy.loc[flags.any(axis=1)])

### Rolling Z-score: compare a return with PRIOR observations
Shift the rolling mean and standard deviation by one observation so today's suspected outlier does not inflate its own threshold.
Undefined warm-up scores and zero-variance history remain `NaN`. These statistical flags request review; they do not establish an error.
A spike followed by a reversal requires tomorrow's data, so it is only a retrospective diagnostic.

In [ ]:
def rolling_zscore(series, window=20):
    """Score against a complete trailing window excluding the current observation."""
    history = series.shift(1)
    mean = history.rolling(window, min_periods=window).mean()
    std = history.rolling(window, min_periods=window).std().replace(0, np.nan)
    return (series - mean) / std

returns = messy['close'].pct_change(fill_method=None)
z = rolling_zscore(returns)
review = pd.DataFrame({'extreme_return': returns.abs() > 0.20, 'rolling_z': z.abs() > 3,
                       'spike_reversal': (returns.abs() > 0.15) & (returns.shift(-1).abs() > 0.15)
                       & (returns * returns.shift(-1) < 0)})
display(review.sum())
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(messy.index, messy['close'], label='Close')
mask = flags.any(axis=1) | review.any(axis=1)
ax.scatter(messy.index[mask], messy.loc[mask, 'close'], color='red', label='Investigate')
ax.set(title='Bad price versus unusual return', ylabel='Price', xlabel='Date'); ax.legend()
plt.show()

### Repair known faults and leave an audit trail
For this controlled example, same-day open/high/low are trustworthy. Use their range midpoint for the two damaged closes and mark them.
In real vendor data, identify the faulty field before deciding which source to trust; a bounds violation alone does not identify it.
Zero is a documented placeholder for invalid volume. Whole-series median fill and interpolation are retrospective alternatives in the reference notebook.

In [ ]:
cleaned = messy.copy()
close_bad = (~np.isfinite(cleaned['close']) | (cleaned['close'] <= 0)
             | ~cleaned['close'].between(cleaned['low'], cleaned['high']))
volume_bad = ~np.isfinite(cleaned['volume']) | (cleaned['volume'] < 0)
cleaned['close_repaired'] = close_bad
cleaned['volume_repaired'] = volume_bad
cleaned.loc[close_bad, 'close'] = (cleaned.loc[close_bad, 'high'] + cleaned.loc[close_bad, 'low']) / 2
cleaned.loc[volume_bad, 'volume'] = 0
assert not quality_flags(cleaned).any().any()
quality_report = flags.rename_axis('issue_date').reset_index().melt(
    id_vars='issue_date', var_name='issue_type', value_name='flagged')
quality_report = quality_report.loc[quality_report['flagged']].drop(columns='flagged')
quality_report['ticker'] = ticker
display(quality_report)
# Retain original, messy, cleaned, and repair flags to explain every change.
print('All deterministic checks passed; review flags remain for investigation.')

### Write and read back a demonstration table
Practice `to_sql` in a separate local database. The bCourses source remains read-only.
The report template uses the supplied `ohlc` table; homework builds `ohlc_hw_cleaned` from `ohlc_hw` independently.

In [ ]:
output_dir = lesson_dir / 'outputs'
output_dir.mkdir(exist_ok=True)
with sqlite3.connect(output_dir / 'cleaning_demo.db') as demo_conn:
    cleaned.reset_index().to_sql('cleaned_demo', demo_conn, if_exists='replace', index=False)
    roundtrip = pd.read_sql_query('SELECT * FROM cleaned_demo ORDER BY ts', demo_conn, parse_dates=['ts'])
assert len(roundtrip) == len(cleaned)
assert np.allclose(roundtrip['close'], cleaned['close'])
quality_report.to_csv(output_dir / 'quality_report.csv', index=False)

## 4. Ten-minute homework query exercise
Write a query counting rows and missing closes per ticker in `ohlc_hw`. Use `COUNT(*) - COUNT(close)`.
Then explain why a missing **row** is different from a null **cell**.
The worked answer below inspects homework data without cleaning it for you.

In [ ]:
# Write your query here

In [ ]:
hw_summary = pd.read_sql_query("""
SELECT ticker, COUNT(*) AS rows, COUNT(*) - COUNT(close) AS missing_closes,
       MIN(ts) AS first_date, MAX(ts) AS last_date
FROM ohlc_hw GROUP BY ticker ORDER BY ticker
""", conn)
display(hw_summary.head(10))
conn.close()
print('Source database connection closed.')

**Missing dates:** compare each ticker with a documented expected trading calendar during its active history.
A `bdate_range` includes exchange holidays; it is only a weekday approximation, and listings/delistings also matter.

**Break: 10:25-10:35.** Next: [one stock report](2_stock_analysis_template.ipynb).
Optional extensions: [Prophet and outlier methods](reference/1_advanced_outlier_detection.ipynb),
[full multi-asset cleaning case](reference/3_real_world_data_challenge.ipynb).